# 🚀 Sample-Level 3D Dataset Generation

Generates one row per `(sample, layer)`:
- `D`: mean-pooled hidden state of `x1` at layer `l`
- `y1`: mean-pooled softmax router scores of `x1` at layer `l`
- `y2`: mean-pooled softmax router scores of `x2` at layer `l`

Prompts:
- `x1 = "Document {context} Question {question}"`
- `x2 = "Question {question}"`

Output file: `dataset.pt` with `{"D", "y1", "y2", "metadata"}`.

## 1 · Environment Setup

In [24]:
%cd /home1/kelidari/llm-steering

/home1/kelidari/llm-steering


In [25]:
import os
import sys

miniconda_path = f"{os.environ.get('HOME', '')}/miniconda/bin"
os.environ["PATH"] = f"{miniconda_path}:" + os.environ.get("PATH", "")

SCRATCH = os.environ.get("SCRATCH_DIR", "/scratch1/kelidari")
os.environ["HF_HOME"] = f"{SCRATCH}/hf_cache"
os.environ["TMPDIR"] = f"{SCRATCH}/tmp"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

print(f"Conda PATH = {miniconda_path}")
print(f"HF_HOME    = {os.environ['HF_HOME']}")
print(f"TMPDIR     = {os.environ['TMPDIR']}")

Conda PATH = /home1/kelidari/miniconda/bin
HF_HOME    = /scratch1/kelidari/hf_cache
TMPDIR     = /scratch1/kelidari/tmp


In [26]:
# import os

env_name = "llm_steering"
# print("Checking environment status...")
# res = os.system(f"conda env list | grep {env_name} > /dev/null")
# if res == 0:
#     print(f"{env_name} exists! Synchronizing any missing packages ...")
#     os.system("conda env update -f environment.yml --prune")
# else:
#     print(f"{env_name} does not exist. Creating fresh environment...")
#     os.system("conda env create -f environment.yml")

## 2 · Configuration

In [27]:
import os

MODEL_NAME = "allenai/OLMoE-1B-7B-0125-Instruct"
DEVICE = "cuda"
STORAGE_DTYPE = "float16"
MAX_EXAMPLES = 5  # Set to 5 for a quick manual smoke run.
CHECKPOINT_INTERVAL = 100
SPLIT = "train"

SCRATCH = os.environ.get("SCRATCH_DIR", "/scratch1/kelidari")
OUTPUT_DIR = f"{SCRATCH}/dataset_3d/output"
CHECKPOINT_DIR = f"{SCRATCH}/dataset_3d/checkpoints"
DATASET_FILE = f"{OUTPUT_DIR}/dataset.pt"
os.environ['WANDB_API_KEY'] = "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"


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Model          = {MODEL_NAME}")
print(f"Split          = {SPLIT}")
print(f"Storage dtype  = {STORAGE_DTYPE}")
print(f"Output file    = {DATASET_FILE}")
print(f"Checkpoint dir = {CHECKPOINT_DIR}")
print(f"Max examples   = {MAX_EXAMPLES or 'all (~87K)'}")
print("✓ W&B API key configured!")

Model          = allenai/OLMoE-1B-7B-0125-Instruct
Split          = train
Storage dtype  = float16
Output file    = /scratch1/kelidari/dataset_3d/output/dataset.pt
Checkpoint dir = /scratch1/kelidari/dataset_3d/checkpoints
Max examples   = 5
✓ W&B API key configured!


In [28]:
# OLMoE-1B-7B-0125-Instruct defaults
L = 16
E = 64
D = 2048
S = MAX_EXAMPLES or 87599  # SQuAD train size
rows = S * L
bytes_per_element = 2 if STORAGE_DTYPE == "float16" else 4
per_row_bytes = (D + 2 * E) * bytes_per_element
total_gb = rows * per_row_bytes / 1e9

print(f"Rows (S*L):          {rows:,}")
print(f"Per-row bytes:       {per_row_bytes:,}")
print(f"Estimated dataset:   {total_gb:.2f} GB ({STORAGE_DTYPE})")
print("Formula: S * L * (D + 2E) * bytes_per_element")

Rows (S*L):          80
Per-row bytes:       4,352
Estimated dataset:   0.00 GB (float16)
Formula: S * L * (D + 2E) * bytes_per_element


In [29]:
%%writefile check_gpu.py
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"torch={torch.__version__}")
print(f"torch.version.cuda={torch.version.cuda}")
print(f"cuda.is_available={torch.cuda.is_available()}")

if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA requested but unavailable. Run this in a GPU allocation and use a CUDA-enabled torch build."
    )

print(f"Running on: {DEVICE}")

Overwriting check_gpu.py


In [30]:
!conda run -n llm_steering python -u -m check_gpu

torch=2.10.0+cu128
torch.version.cuda=12.8
cuda.is_available=True
Running on: cuda


## 3 · Run Dataset Generation

In [32]:
MAX_EXAMPLES_ARG = f"--max-examples {MAX_EXAMPLES}" if MAX_EXAMPLES is not None else ""
!conda run -n llm_steering python -u -m src.dataset_3d.generate \
    --model "{MODEL_NAME}" \
    --output-dir "{OUTPUT_DIR}" \
    --checkpoint-dir "{CHECKPOINT_DIR}" \
    --checkpoint-interval {CHECKPOINT_INTERVAL} \
    --storage-dtype "{STORAGE_DTYPE}" \
    --split "{SPLIT}" \
    --device "{DEVICE}" {MAX_EXAMPLES_ARG}

W&B run initialized
Loading tokenizer: allenai/OLMoE-1B-7B-0125-Instruct
Loading SQuAD split=train...
  Samples: 5
Loading model: allenai/OLMoE-1B-7B-0125-Instruct
  Model input device: cuda:0
Resuming from sample 5/5
Estimated dataset size (float16): 0.00 GB for rows=80
  checkpoint saved: sample_idx=5/5
Saved dataset to: /scratch1/kelidari/dataset_3d/output/dataset.pt
D shape:  (80, 2048)
y1 shape: (80, 64)
y2 shape: (80, 64)
Total time: 1.9 minutes
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kelidari (AGSER) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run sx4myybh
wandb: Tracking run with wandb version 0.26.0
wandb: Run data is saved locally in /home1/kelidari/llm-steering/wandb/run-20260415_163659-sx4myybh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sample-level-OLMoE-1B-7B-0125-Instruct-5ex
wandb: ⭐️ View project at https://wandb.ai/VLAvenger

## 4 · Verification

In [ ]:
%%writefile verify_dataset.py
import sys
import torch

dataset_path = sys.argv[1]
obj = torch.load(dataset_path, map_location='cpu', weights_only=False)
D, y1, y2, meta = obj['D'], obj['y1'], obj['y2'], obj['metadata']
rows = meta['rows']
hidden_dim = meta['hidden_dim']
experts = meta['experts']
print(f'Loaded: {dataset_path}')
print(f'D shape:  {tuple(D.shape)}')
print(f'y1 shape: {tuple(y1.shape)}')
print(f'y2 shape: {tuple(y2.shape)}')
print(f"Metadata rows={rows}, D={hidden_dim}, E={experts}, L={meta['layers']}, S={meta['num_samples']}")
assert D.shape == (rows, hidden_dim)
assert y1.shape == (rows, experts)
assert y2.shape == (rows, experts)
assert D.dtype == y1.dtype == y2.dtype
assert not torch.isnan(D.float()).any()
assert not torch.isnan(y1.float()).any()
assert not torch.isnan(y2.float()).any()
sum_y1 = y1.float().sum(dim=-1)
sum_y2 = y2.float().sum(dim=-1)
print(f'y1 row-sum mean: {sum_y1.mean().item():.6f}')
print(f'y2 row-sum mean: {sum_y2.mean().item():.6f}')
print('Verification checks passed.')

In [ ]:
!conda run -n llm_steering python -u verify_dataset.py "{DATASET_FILE}"

## 5 · Merge Shards

After all 32 parallel SLURM jobs finish, run these cells to merge the
shard files into a single `dataset.pt`.  
**Run on a CPU node with ≥128 GB RAM (no GPU needed).**

In [ ]:
SHARD_DIR   = f"{SCRATCH}/dataset_3d/shards"
MERGED_FILE = f"{SCRATCH}/dataset_3d/output/dataset.pt"
NUM_SHARDS  = 32

print(f"Shard dir:   {SHARD_DIR}")
print(f"Merged file: {MERGED_FILE}")
print(f"Num shards:  {NUM_SHARDS}")

In [ ]:
%%writefile merge_shards.py
"""Merge shard_XX.pt files into a single dataset.pt."""
import sys
import os
import torch

shard_dir   = sys.argv[1]
output_path = sys.argv[2]
num_shards  = int(sys.argv[3])

D_parts, y1_parts, y2_parts = [], [], []
total_samples = 0
meta = None

for i in range(num_shards):
    path = os.path.join(shard_dir, f'shard_{i:02d}.pt')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing shard: {path}')
    print(f'Loading shard {i:02d}/{num_shards}: {path}')
    obj = torch.load(path, map_location='cpu', weights_only=False)
    D_parts.append(obj['D'])
    y1_parts.append(obj['y1'])
    y2_parts.append(obj['y2'])
    total_samples += obj['metadata']['num_samples']
    if meta is None:
        meta = obj['metadata'].copy()

print(f'\nConcatenating {num_shards} shards...')
D  = torch.cat(D_parts,  dim=0)
y1 = torch.cat(y1_parts, dim=0)
y2 = torch.cat(y2_parts, dim=0)
del D_parts, y1_parts, y2_parts   # free shard memory

# Update metadata for merged dataset
meta['rows']        = D.shape[0]
meta['num_samples'] = total_samples
meta.pop('shard_index', None)
meta.pop('num_shards', None)

print(f'D shape:  {tuple(D.shape)}')
print(f'y1 shape: {tuple(y1.shape)}')
print(f'y2 shape: {tuple(y2.shape)}')
print(f'Total samples: {total_samples}')
print(f'Total rows:    {D.shape[0]}')

os.makedirs(os.path.dirname(output_path), exist_ok=True)
tmp_path = output_path + '.tmp'
torch.save({'D': D, 'y1': y1, 'y2': y2, 'metadata': meta}, tmp_path)
os.replace(tmp_path, output_path)
print(f'\nMerged dataset saved to: {output_path}')

In [ ]:
!conda run -n llm_steering python -u merge_shards.py \
    "{SHARD_DIR}" \
    "{MERGED_FILE}" \
    {NUM_SHARDS}

In [ ]:
# Re-use the verification script on the merged dataset
!conda run -n llm_steering python -u verify_dataset.py "{MERGED_FILE}"

## 6 · Train 3D DELTA (Rank Regression)

Train the rank regression DELTA (Pairwise Margin Loss) on the generated dataset.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
MARGIN = 0.05
TRAIN_DATASET_FILE = DATASET_FILE if MAX_EXAMPLES else MERGED_FILE
TRAINED_DELTA_FILE = f"{OUTPUT_DIR}/trained_delta.pt"

print(f"Training Dataset: {TRAIN_DATASET_FILE}")
print(f"Output DELTA:     {TRAINED_DELTA_FILE}")

In [ ]:
!conda run -n llm_steering python -u src/dataset_3d/train_delta.py \
    --dataset-path "{TRAIN_DATASET_FILE}" \
    --output-path "{TRAINED_DELTA_FILE}" \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --margin {MARGIN}

### Verify Trained DELTA

In [ ]:
import torch
delta = torch.load(TRAINED_DELTA_FILE, map_location="cpu", weights_only=False)
print(f"Trained DELTA shape: {tuple(delta.shape)}")
assert delta.dim() == 3, "Expected 3D tensor"